# 🎯 [PRC01_M07] 계약 승인 전 법무/계약관 수동 검토 및 조항 수정 권고 HITL 절차 설계 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M07 |
| **미션 주제** | 계약 승인 전 법무/계약관 수동 검토 및 조항 수정 권고 HITL 절차 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 계약 승인 전 법무/계약관 수동 검토 및 조항 수정 권고 HITL 절차 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

계약 승인 전 계약관 수동 검토 및 조항 수정 권고 HITL 절차를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **계약 승인 전 계약관·법무 수동 검토와 조항 수정 권고를 위한 HITL 절차를 설계하고 이력 무결성과 권한 통제를 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **시스템 제안, 계약담당 상신, 계약관 검토, 법무 검토, 승인권자 결재의 순서를 상태 기계로 구현하고 역할별 권한을 최소로 둔다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/workflow/prc01_approval_system.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 보안 규정 및 행정 절차 부합성

#### 미션 PASS 조건

- 정량 기준: 이력 기록 무결성 100%, 승인 권한 우회 시도 차단율 100%


- 진행 기준: 결재선 승인/반려 시뮬레이션 및 이력 조회 화면 구현


- 검증 방법: 권한별 접근 제어(RBAC) 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""고위험 계약의 조항 수정 권고를 계약담당 상신 -> 계약관 검토 -> 법무 검토 -> 승인권자 결재의 HITL 워크플로우로 처리하되, 반려 사유 필수화·재기안 연계·시스템 자동 확정 금지·권한 우회 차단(RBAC)과 해시 체인 감사 로그의 이력 무결성·위변조 탐지를 검증한다."""
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 조달계획안·계약서 생성기 (실제 계약·업체·법령 조문 없음)
# 계약 속성 16개 + 조항 문장 6개를 만들고, 속성 위반·독소 조항·납기 위험 문구가 섞인 '규정 위반 여부'를 함께 만든다.
# ---------------------------------------------------------------------------
TYPES = ["물품", "용역", "공사"]
METHODS = ["일반경쟁", "제한경쟁", "지명경쟁", "수의계약"]
CLASSES = ["정상", "독소", "납기위험"]
OBJ = ["납품 물품", "용역 결과물", "시공 목적물", "정비 성과물"]

N_TPL = [
    "계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 {p}/1000에 해당하는 지체상금을 부과한다.",
    "지체상금은 불가항력 사유가 확인된 경우에 한하여 발주기관의 승인을 받아 면제할 수 있다.",
    "납기 연장은 발주기관이 불가항력 사유를 확인하고 서면으로 승인한 경우에만 인정한다.",
    "발주기관은 납품된 {o}을 {d}일 이내에 검사하고 합격한 후에 대금을 지급한다.",
    "계약상대자는 하도급을 하려는 경우 사전에 발주기관의 서면 승인을 받아야 한다.",
    "어느 한쪽이 계약을 해지하려면 사유를 명시해 {d}일 전에 서면으로 통지해야 한다.",
    "{o}에 하자가 발생하면 계약상대자는 보증 기간 안에 무상으로 보수하거나 교체한다.",
    "계약 이행 중 알게 된 비공개 사항은 제3자에게 누설하지 않으며 위반 시 책임을 진다.",
    "{o}에 대한 지식재산권의 귀속과 사용 범위는 별도 합의서에 따라 정한다.",
    "대금은 검사 합격 후 청구서를 접수한 날부터 {d}일 이내에 지급한다.",
    "계약금액의 조정은 물가변동 기준에 따라 양 당사자가 서면으로 합의한 경우에만 한다.",
    "계약상대자는 이행보증서를 계약 체결 시 제출하며 보증금은 계약 이행 완료 후 반환한다.",
    "납품 일정은 계약서 별표의 일자를 따르며 변경 시 서면 합의가 필요하다.",
    "지체상금은 면제 사유가 없는 한 부과하며 이의는 정해진 절차로 제기할 수 있다.",
    "검사에 합격하지 못한 {o}은 계약상대자가 교체하며 그 비용은 계약상대자가 부담한다.",
    "계약상대자의 귀책 사유로 이행이 불가능하면 발주기관은 서면으로 통지하고 계약을 해지할 수 있다.",
    "지연이 예상되면 계약상대자는 사유와 만회 계획을 발주기관에 서면으로 보고한다.",
    "보증 기간 중 발생한 결함은 계약상대자가 책임지고 보수한다.",
    "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
]
T_TPL = [
    "계약상대자의 지체상금은 전액 면제하며 발주기관은 이에 대해 이의를 제기하지 못한다.",
    "납품 즉시 검사에 합격한 것으로 보며 발주기관은 이후 검사를 요구하지 않는다.",
    "계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "{o}의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이 청구할 수 있다.",
    "{o}에 관한 지식재산권은 모두 계약상대자에게 귀속되고 발주기관은 사용할 수 없다.",
    "하도급은 신고 없이 제한 없이 허용하며 발주기관은 하도급 내용을 확인하지 않는다.",
    "대금은 검사 전에 전액 선지급하며 발주기관은 반환 청구권을 포기한다.",
    "계약 이행 중 알게 된 사항의 누설로 인한 책임을 계약상대자는 지지 않는다.",
    "하자 보수 책임은 인도 후 즉시 소멸하며 발주기관은 보수를 요구할 수 없다.",
    "발주기관은 계약상대자가 청구하는 금액을 거절할 수 없으며 거절하면 위약금을 지급한다.",
    "분쟁이 생기면 계약상대자가 정하는 중재 기관에서만 해결하고 발주기관은 따른다.",
    "발주기관의 검사는 생략하고 계약상대자의 시험 결과서로 대신한다.",
    "견적 금액은 사후 정산 없이 그대로 확정하고 증빙 자료는 받지 않는다.",
]
D_TPL = [
    "납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.",
    "납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다.",
    "공급 지연이나 선행 업체 사정도 불가항력으로 보아 납기를 자동 연장한다.",
    "착수일은 선행 업체의 납품이 끝난 뒤 별도로 정하며 완료 기한은 정하지 않는다.",
    "납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
    "납기는 계약상대자의 생산 일정이 확정되는 대로 통보한다.",
    "납품 시점은 계약상대자 사정이 허락하는 때로 하고 날짜는 따로 정하지 않는다.",
    "자재가 늦게 들어오면 완료 일정은 그만큼 자동으로 늦춰진다.",
    "기한 내에 인도하지 못해도 계약상대자에게 책임을 묻지 않는다.",
    "일정이 늦어져도 계약상대자는 새 계획서를 낼 필요가 없다.",
]
# 학습에 쓰지 않는 '처음 보는 표현'(모델이 문장 틀을 외웠는지 확인하는 용도)
HOLD = {
    "정상": ["검사에 불합격한 경우 계약상대자는 지정 기한 안에 재납품하며 재검사 비용은 상대자가 부담한다.",
           "계약상대자가 법령을 위반하여 계약을 이행하지 못하면 발주기관은 서면 통지 후 계약을 해지할 수 있다.",
           "납기가 지연될 우려가 있으면 상대자는 즉시 사유와 만회 계획을 서면으로 보고해야 한다.",
           "하자담보 책임 기간은 인수일부터 기산하며 기간 중 발생한 결함은 상대자가 책임진다."],
    "독소": ["발주기관은 계약상대자가 요구하는 대금을 이유 없이 거절하지 못하며 거절 시 위약금을 낸다.",
           "계약 내용에 관한 분쟁은 계약상대자가 지정하는 기관에서만 다루며 발주기관은 이에 따른다.",
           "발주기관의 검수 권한은 포기하고 계약상대자의 자체 시험 성적서로 갈음한다.",
           "계약상대자가 제출한 견적은 사후 정산 없이 확정 단가로 하며 증빙은 요구하지 않는다."],
    "납기위험": ["납품 시기는 상대자의 형편이 허락하는 시점으로 하고 별도 날짜는 정하지 않는다.",
             "원자재 확보가 늦어지면 완료 일자는 자동으로 뒤로 미뤄지는 것으로 본다.",
             "인도 기한은 참고용이며 지키지 못해도 책임을 묻지 않는다.",
             "일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다."],
}
SUFFIX = ["", "", "", " 세부 사항은 시행 세칙에 따른다.", " 이 조항은 계약 전체에 적용한다.", " 필요한 경우 양 당사자가 협의한다.", " 관련 서식은 별지를 따른다."]
SWAP = [("발주기관", "계약담당관"), ("계약상대자", "업체")]


def make_clause(rng, cls, hold=False, idx=None):
    """조항 한 문장 생성. 표현 일부는 동의어로 바꾸고 접미 문구를 무작위로 붙인다(idx를 주면 그 번호의 문장 틀을 사용)."""
    if idx is not None:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[idx].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    elif hold:
        s = HOLD[cls][rng.integers(len(HOLD[cls]))]
    else:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[rng.integers(len(tpl))].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    if rng.random() < 0.3:
        a, b = SWAP[rng.integers(2)]
        s = s.replace(a, b)
    return s + SUFFIX[rng.integers(len(SUFFIX))]


def make_contracts(n=200, seed=7):
    """가상 계약 n건(속성 16개 + 조항 6개 + 규정 위반 정답)을 만든다."""
    rng = np.random.default_rng(seed)
    sl = rng.beta(1.2, 3.2, n)                                  # 계약서 작성이 허술한 정도(잠재 변수)
    ctype = rng.choice(TYPES, n, p=[.5, .3, .2])
    amount = np.clip(np.exp(rng.normal(np.log(220), 1.1, n)), 5, 5000).round(0)
    method = np.where(rng.random(n) < .06 + .3 * sl, "수의계약", rng.choice(METHODS[:3], n, p=[.5, .3, .2]))
    period = np.array([rng.integers(60, 300) if t == "물품" else rng.integers(90, 400) if t == "용역" else rng.integers(180, 720) for t in ctype])
    delivery = (period * rng.uniform(.5, 1.0, n)).astype(int)
    bad_logic = rng.random(n) < .015 * (1 + 3 * sl)
    delivery = np.where(bad_logic, period + rng.integers(10, 60, n), delivery)
    pen = np.where(rng.random(n) < .22 * sl, rng.choice([.3, .5, .8], n), rng.choice([1.0, 1.5, 2.0, 2.5], n))
    adv = np.where(rng.random(n) < .22 * sl, rng.choice([50, 60, 70], n), rng.choice([0, 10, 20, 30, 40], n))
    bond = np.where(rng.random(n) < .25 * sl + .5 * (adv > 40), rng.choice([0, 5], n), rng.choice([10, 10, 15, 20], n))
    war = np.array([rng.choice([3, 6]) if (rng.random() < .25 * s and t == "물품") else rng.choice([12, 24, 36]) if t == "물품" else rng.choice([0, 6, 12]) if t == "용역" else rng.choice([24, 36, 60]) for t, s in zip(ctype, sl)])
    insp = np.where(rng.random(n) < .3 * sl, rng.choice([21, 30], n), rng.choice([5, 7, 10, 14], n))
    sub = np.where(rng.random(n) < .15 * sl, rng.choice([60, 70, 80], n), rng.choice([0, 10, 20, 30, 40, 50], n))
    padj = rng.random(n) < .75 - .4 * sl
    term = rng.random(n) > .015 * (1 + 6 * sl)
    sec = rng.random(n) > .015 * (1 + 6 * sl)
    ip = rng.random(n) > .03 * (1 + 4 * sl)
    clauses, cls_true = [], []
    for i in range(n):
        pt, pd_ = .01 + .08 * sl[i], .02 + .08 * sl[i]
        cl = [rng.choice(CLASSES, p=[1 - pt - pd_, pt, pd_]) for _ in range(6)]
        cls_true.append(cl); clauses.append([make_clause(rng, c) for c in cl])
    df = pd.DataFrame({"contract_id": [f"C-{i + 1:04d}" for i in range(n)], "ctype": ctype, "method": method, "amount_m": amount,
                       "period_days": period, "delivery_days": delivery, "penalty_permil": pen, "advance_pct": adv, "perf_bond_pct": bond,
                       "warranty_months": war, "inspection_days": insp, "subcontract_pct": sub, "price_adjust": padj,
                       "termination": term, "security": sec, "ip_clause": ip})
    hard = ((pen < 1.0) | ((adv > 40) & (bond < 10)) | ((amount >= 300) & (method == "수의계약")) | ((ctype == "물품") & (war < 12))
            | ~term | ~sec | (sub > 50) | (delivery > period))
    toxic = np.array([c.count("독소") for c in cls_true])
    delay = np.array([c.count("납기위험") for c in cls_true])
    split = (method == "수의계약") & (amount >= 250) & (amount < 300) & (rng.random(n) < .6)      # 계약 쪼개기 의심(임계값 바로 아래)
    df["clauses"], df["clause_cls"] = clauses, cls_true
    df["n_toxic"], df["n_delay"] = toxic, delay
    viol = hard | (toxic > 0) | split | (delay >= 2)
    waived = hard & (toxic == 0) & (rng.random(n) < .12)                  # 계약관이 예외 사유를 인정해 위반으로 보지 않은 건(자료의 현실적 잡음)
    hidden = ~viol & (rng.random(n) < .03 + .08 * sl)                       # 속성·문구에 드러나지 않는 위반(별첨 합의 등)
    df["violation"] = (viol & ~waived) | hidden
    df["vtype"] = np.where(hidden, "기타", np.where(hard, "속성위반", np.where(toxic > 0, "독소조항", np.where(split | (delay >= 2), "기타", "-"))))
    return df

# ---- 계약 규정 준수 판별 룰 설정(YAML 형식 문자열: 규정이 바뀌면 코드가 아니라 이 설정만 고친다) ----
RULES_YAML = """
- id: R01
  name: 지체상금률 1.0/1000 미만
  field: penalty_permil
  op: "<"
  value: 1.0
  weight: 3
- id: R02
  name: 선금 비율 40% 초과
  field: advance_pct
  op: ">"
  value: 40
  weight: 2
- id: R03
  name: 선금 40% 초과인데 이행보증 10% 미만
  field: adv_no_bond
  op: "=="
  value: true
  weight: 3
- id: R04
  name: 300백만원 이상 수의계약
  field: sole_amount
  op: ">="
  value: 300
  weight: 3
- id: R05
  name: 500백만원 이상 이행보증 10% 미만
  field: bond_gap
  op: "=="
  value: true
  weight: 2
- id: R06
  name: 물품 하자보증 12개월 미만
  field: warranty_gap
  op: "=="
  value: true
  weight: 2
- id: R07
  name: 해지 조항 없음
  field: termination
  op: "=="
  value: false
  weight: 2
- id: R08
  name: 보안 조항 없음
  field: security
  op: "=="
  value: false
  weight: 3
- id: R09
  name: 지식재산 조항 없음(물품·용역)
  field: ip_missing
  op: "=="
  value: true
  weight: 1
- id: R10
  name: 하도급 비율 50% 초과
  field: subcontract_pct
  op: ">"
  value: 50
  weight: 3
- id: R11
  name: 검사 기간 14일 초과
  field: inspection_days
  op: ">"
  value: 14
  weight: 1
- id: R12
  name: 납품 기한이 계약 기간 초과
  field: logic_err
  op: "=="
  value: true
  weight: 3
- id: R13
  name: 180일 이상 계약에 물가변동 조항 없음
  field: no_padj_long
  op: "=="
  value: true
  weight: 1
- id: R14
  name: 1000백만원 이상 비경쟁 계약
  field: noncomp_big
  op: "=="
  value: true
  weight: 2
- id: R15
  name: 계약 기간 365일 초과
  field: period_days
  op: ">"
  value: 365
  weight: 1
- id: R16
  name: 불공정·납기 위험 키워드 포함
  field: kw_hits
  op: ">="
  value: 1
  weight: 3
"""
GRADE_CUT = {"고위험": 6, "주의": 3}          # 가중치 합 기준 1차 판별 등급(필요 시 설정만 바꾼다)
# [교육용가상자료] 불공정 조항·납기 지체 위험 문구 사전(실제 규정 조문이 아님)
KEYWORDS = {"독소": ["면제하며", "일방적으로", "포기", "지지 않는다", "즉시 소멸", "제한 없이", "신고 없이", "전액 부담", "별도 협의 없이", "검사 전에", "검사에 합격한 것으로", "사용할 수 없다", "증빙은 요구하지 않는다", "지정하는 기관"],
            "납기위험": ["자동 연장", "기한 없이", "정하지 않는다", "노력하며", "연장할 수 있다", "통보한다", "책임을 묻지 않는다", "의무가 없다"]}


def parse_rules(text):
    """간이 YAML 파서(목록 + key: value). 숫자·true/false·따옴표 문자열을 해석한다."""
    # [TODO : 학생 실습 7] 계약 승인 전 법무/계약관 수동 검토 및 조항 수정 권고 HITL 절차 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "<": lambda a, b: a < b, "<=": lambda a, b: a <= b}


def kw_count(clauses):
    """조항 문장 목록에서 독소·납기 위험 키워드가 들어 있는 조항 수."""
    # [TODO : 학생 실습 7] 계약 승인 전 법무/계약관 수동 검토 및 조항 수정 권고 HITL 절차 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def derive(df):
    """룰에 쓰는 파생 필드(선금+보증 부족, 수의계약 고액, 논리 오류, 키워드 적중 등)를 만든다."""
    d = df.copy()
    d["adv_no_bond"] = (d["advance_pct"] > 40) & (d["perf_bond_pct"] < 10)
    d["sole_amount"] = np.where(d["method"] == "수의계약", d["amount_m"], 0.0)       # 수의계약일 때만 금액(한도는 YAML에서 조정)
    d["bond_gap"] = (d["amount_m"] >= 500) & (d["perf_bond_pct"] < 10)
    d["warranty_gap"] = (d["ctype"] == "물품") & (d["warranty_months"] < 12)
    d["ip_missing"] = (~d["ip_clause"]) & (d["ctype"] != "공사")
    d["logic_err"] = d["delivery_days"] > d["period_days"]
    d["no_padj_long"] = (d["period_days"] >= 180) & (~d["price_adjust"])
    d["noncomp_big"] = d["method"].isin(["지명경쟁", "수의계약"]) & (d["amount_m"] >= 1000)
    d["kw_hits"] = [kw_count(c) for c in d["clauses"]] if "clauses" in d else 0
    return d


def apply_rules(rec, rules):
    """레코드 1건(dict)에 룰을 적용해 (점수, 등급, 적중 룰 id 목록)을 돌려준다. 값이 없으면 해당 룰은 건너뛴다."""
    hits = []
    for r in rules:
        v = rec.get(r["field"])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if OPS[r["op"]](v, r["value"]):
            hits.append(r)
    score = sum(r["weight"] for r in hits)
    grade = "고위험" if score >= GRADE_CUT["고위험"] else ("주의" if score >= GRADE_CUT["주의"] else "관심")
    return score, grade, [r["id"] for r in hits]

SIMPLE_KW = {"독소": ["면제", "일방", "포기", "전액", "누설", "하도급", "선지급"], "납기위험": ["연장", "자동", "노력", "통보", "기한"]}

# ---- 조항 문장 분류(정상 / 독소 / 납기위험): 글자 n-gram TF-IDF + 로지스틱 회귀 ----
def clause_set(seed, n, hold=False, mix=(.55, .25, .20)):
    """조항 문장 n개와 정답 분류를 만든다. hold=True면 학습에 쓰지 않은 '처음 보는 표현'으로 만든다."""
    rng = np.random.default_rng(seed)
    y = rng.choice(CLASSES, n, p=list(mix))
    return [make_clause(rng, c, hold) for c in y], list(y)


def fit_clause_model(seed=100, n=1200):
    """학습용 조항 문장으로 분류 모델을 학습한다."""
    X, y = clause_set(seed, n)
    return make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True), LogisticRegression(C=20, max_iter=2000)).fit(X, y)


def keyword_pred(s):
    """비교용 기준선: 단순 키워드(부정 표현·문맥을 보지 않음) 적중으로만 분류."""
    if any(k in s for k in SIMPLE_KW["독소"]):
        return "독소"
    return "납기위험" if any(k in s for k in SIMPLE_KW["납기위험"]) else "정상"

# ---- 역할과 권한(최소 권한 원칙): 시스템은 '제안'만 할 수 있고 확정 권한이 없다 ----
PERM = {"시스템": {"propose"}, "계약담당": {"submit", "redraft", "view"}, "계약관": {"review_contract", "reject", "view"}, "법무": {"review_legal", "reject", "view"},
        "승인권자": {"approve", "reject", "view"}, "감사관": {"view", "audit"},
        "부서장": {"submit", "approve", "reject", "view"}}            # 소규모 부서는 상신과 결재를 겸할 수 있으나 같은 건은 분리
NEXT = {"제안": ("submit", "상신"), "상신": ("review_contract", "계약관검토"), "계약관검토": ("review_legal", "법무검토"), "법무검토": ("approve", "승인")}
USERS = {"sys": "시스템", "kim": "계약담당", "lee": "계약관", "park": "법무", "choi": "승인권자", "jung": "감사관", "oh": "부서장"}
MAX_REDRAFT = 2


class Chain:
    """해시 체인 감사 로그: 각 기록이 직전 기록의 해시를 포함해 중간 수정이 드러난다."""

    def __init__(self):
        self.log = []

    def add(self, **body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), **body}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            b = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(b, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return e["seq"]
            prev = e["hash"]
        return None


class Workflow:
    def __init__(self):
        self.audit, self.cases, self.blocked = Chain(), {}, []

    def _deny(self, who, case, action, why):
        self.audit.add(event="DENIED", who=who, case=case, action=action, reason=why)
        self.blocked.append((who, case, action, why)); return False, why

    def act(self, who, case, action, reason="", evidence=""):
        role = USERS.get(who)
        if role is None:
            return self._deny(who, case, action, "미등록 사용자")
        if action == "delete_log":
            return self._deny(who, case, action, "감사 로그 삭제·수정 권한은 누구에게도 없음")
        if action not in PERM[role]:
            return self._deny(who, case, action, f"권한 없음({role} 역할에 {action} 권한 없음)")
        if action == "propose":
            self.cases[case] = {"state": "제안", "submitter": None, "version": 1}
            self.audit.add(event="PROPOSE", who=who, case=case, reason=reason); return True, "제안 등록"
        c = self.cases.get(case)
        if c is None:
            return self._deny(who, case, action, "존재하지 않는 건")
        if c["state"] in ("승인", "종결(반려)"):
            return self._deny(who, case, action, f"이미 종결된 건({c['state']})")
        if not reason.strip() or not evidence.strip():
            return self._deny(who, case, action, "사유·근거 입력 필수(반려 사유 포함)")
        if action == "reject":
            if c["state"] in ("제안", "반려"):
                return self._deny(who, case, action, f"반려할 수 없는 단계({c['state']})")
            c["state"] = "종결(반려)" if c["version"] > MAX_REDRAFT else "반려"
            self.audit.add(event="REJECTED", who=who, role=role, case=case, version=c["version"], reason=reason, evidence=evidence); return True, c["state"]
        if action == "redraft":
            if c["state"] != "반려":
                return self._deny(who, case, action, f"재기안은 반려된 건만 가능(현재 {c['state']})")
            if who != c["submitter"]:
                return self._deny(who, case, action, "재기안은 최초 상신자만 가능")
            c["state"], c["version"] = "상신", c["version"] + 1
            self.audit.add(event="REDRAFT", who=who, role=role, case=case, version=c["version"], reason=reason, evidence=evidence); return True, "재기안 상신"
        if c["state"] == "반려":
            return self._deny(who, case, action, "반려된 건은 재기안 후에만 진행 가능")
        need, nxt = NEXT[c["state"]]
        if action != need:
            return self._deny(who, case, action, f"단계 위반(현재 {c['state']}, 필요 행위 {need})")
        if action == "submit":
            c["submitter"] = who
        if action == "approve" and who == c["submitter"]:
            return self._deny(who, case, action, "상신자와 승인자 분리 원칙 위반")
        c["state"] = nxt
        self.audit.add(event=action.upper(), who=who, role=role, case=case, version=c["version"], to=nxt, reason=reason, evidence=evidence); return True, nxt


# ---- 상신 대상: 룰 점수가 높은 계약 12건과 모델이 찾은 위험 조항 ----
cm = fit_clause_model()
rules = parse_rules(RULES_YAML)
con = make_contracts(100, seed=7)
d = derive(con)
sc = [apply_rules(r, rules) for r in d.to_dict("records")]
d["score"], d["hits"] = [x[0] for x in sc], [x[2] for x in sc]
picked = d.sort_values("score", ascending=False).head(12)
rng = np.random.default_rng(1)
LEGAL = {"R08", "R07", "R12", "R10"}                      # 법무 검토에서 보는 항목(보안·해지·논리·하도급)

wf = Workflow()
rows = []
for k, (_, p) in enumerate(picked.iterrows(), 1):
    case = f"C{k:02d}"
    flagged = [i + 1 for i, c in enumerate(cm.predict(list(p["clauses"]))) if c != "정상"]
    wf.act("sys", case, "propose", reason=f"{p['contract_id']} 룰 점수 {p['score']}, 위험 조항 {len(flagged)}개 -> 수정 권고 제안")
    wf.act("kim", case, "submit", "위험 조항 수정 권고 검토 요청", f"룰 적중 {','.join(p['hits'][:4])}, 위험 조항 {flagged or '없음'}")
    c_issue, l_issue = bool(flagged), sorted(LEGAL & set(p["hits"]))                # 계약관이 볼 문제, 법무가 볼 문제
    note, big, justified = "승인권자 결정", p["amount_m"] >= 1000 and p["method"] != "일반경쟁", False
    for _ in range(8):
        state = wf.cases[case]["state"]
        if state in ("승인", "종결(반려)"):
            break
        if state == "상신":
            if c_issue:
                wf.act("lee", case, "reject", f"위험 조항 {flagged} 수정 필요", "조항 모델 위험 분류와 계약관 판단 일치")
            else:
                wf.act("lee", case, "review_contract", "조항 검토 완료", f"위험 조항 {'정리됨' if flagged else '없음'}")
        elif state == "계약관검토":
            if rng.random() < 0.15:
                note = "자동 확정 없음: 법무 응답 대기"; break                       # 무응답은 보류로 둔다
            if l_issue:
                wf.act("park", case, "reject", f"필수 조항 보완 필요({','.join(l_issue)})", "해지·보안·하도급·납품 기한 조건 점검")
            else:
                wf.act("park", case, "review_legal", "법무 검토 완료", "필수 조항 점검 이상 없음")
        elif state == "법무검토":
            if big and not justified:
                wf.act("choi", case, "reject", "고액 비경쟁 계약은 사유 소명 후 재상신", f"금액 {p['amount_m']:.0f}백만원, {p['method']}")
            else:
                wf.act("choi", case, "approve", "승인: 수정본으로 계약 의뢰 진행", f"룰 점수 {p['score']}, 위험 조항·필수 조항 정리 완료")
        elif state == "반려":
            wf.act("kim", case, "redraft", "반려 사유 반영해 수정", f"수정 대상 {flagged if c_issue else l_issue or ['소명 자료']}, 재기안 {wf.cases[case]['version']}차")
            if rng.random() < 0.65:                                              # 재기안 후 문제가 정리될 확률(모의)
                c_issue, l_issue = False, []
            justified = True                                                     # 고액 비경쟁 건은 재기안 때 사유 소명서를 첨부(모의)
    final = wf.cases[case]["state"]
    if final not in ("승인", "종결(반려)") and note.startswith("자동"):
        final = "보류(무응답)"
    elif final == "종결(반려)":
        note = "재기안 한도(2회) 초과"
    rows.append((case, p["contract_id"], final, wf.cases[case]["version"], note))
print("[결재 결과] 12건 (모의 결재자 기준, 실제 결재는 사람이 한다)")
print(pd.DataFrame(rows, columns=["건", "계약", "최종 상태", "버전", "비고"]).to_string(index=False))
st = pd.Series([r[2] for r in rows]).value_counts().to_dict()
n_redraft = sum(e["event"] == "REDRAFT" for e in wf.audit.log)
print("상태 집계:", st, f"/ 반려 후 재기안 {n_redraft}회, 반려 이력 {sum(e['event'] == 'REJECTED' for e in wf.audit.log)}건(모두 사유·근거 기록)")

# ---- 권한 우회·절차 위반 시도 ----
wf.act("sys", "T01", "propose", reason="시험 건")
wf.act("kim", "T01", "submit", "시험 상신", "근거")
attempts = [
    ("시스템 자동 승인", lambda: wf.act("sys", "T01", "approve", "자동", "자동")),
    ("계약담당이 계약관 검토 대행", lambda: wf.act("kim", "T01", "review_contract", "대행", "근거")),
    ("검토 단계 건너뛰고 승인", lambda: wf.act("choi", "T01", "approve", "바로 승인", "근거")),
    ("미등록 사용자 접근", lambda: wf.act("guest", "T01", "review_contract", "열람", "근거")),
    ("사유 없는 검토", lambda: wf.act("lee", "T01", "review_contract", "", "")),
    ("사유 없는 반려", lambda: wf.act("lee", "T01", "reject", "", "근거")),
    ("감사 로그 삭제", lambda: wf.act("kim", "T01", "delete_log", "정리", "근거")),
    ("감사관이 승인 시도", lambda: wf.act("jung", "T01", "approve", "승인", "근거")),
    ("종결된 건 재승인", lambda: wf.act("choi", next(c for c, v in wf.cases.items() if v["state"] == "승인"), "approve", "재승인", "근거")),
    ("존재하지 않는 건 승인", lambda: wf.act("choi", "X99", "approve", "승인", "근거")),
]
wf.act("sys", "T02", "propose", reason="시험"); wf.act("oh", "T02", "submit", "상신", "근거")
wf.act("lee", "T02", "review_contract", "조항 확인", "위험 조항 없음"); wf.act("park", "T02", "review_legal", "법무 확인", "이상 없음")
attempts.append(("상신자가 본인 건 승인(분리 원칙)", lambda: wf.act("oh", "T02", "approve", "본인 승인", "근거")))
wf.act("sys", "T03", "propose", reason="시험"); wf.act("kim", "T03", "submit", "상신", "근거"); wf.act("lee", "T03", "reject", "수정 필요", "근거")
attempts.append(("반려 건을 재기안 없이 검토 진행", lambda: wf.act("lee", "T03", "review_contract", "그대로 진행", "근거")))
attempts.append(("타인이 대신 재기안", lambda: wf.act("oh", "T03", "redraft", "대신 수정", "근거")))
print("\n[권한 우회·절차 위반 시도]")
blocked = 0
for name, fn in attempts:
    ok, why = fn()
    blocked += (not ok)
    print(f"  {name:24s}-> {'차단' if not ok else '통과(결함)'}: {why}")
print(f"우회·위반 시도 차단 {blocked}/{len(attempts)} ({blocked / len(attempts):.0%}) (목표 100%: {'달성' if blocked == len(attempts) else '미달'})")

# ---- RBAC 매트릭스(역할별 접근 제어 시험) ----
acts = ["propose", "submit", "review_contract", "review_legal", "approve", "reject", "redraft", "audit"]
mx = pd.DataFrame([[("O" if a in PERM[r] else "-") for a in acts] for r in PERM], index=list(PERM), columns=["제안", "상신", "계약관검토", "법무검토", "승인", "반려", "재기안", "감사조회"])
print("\n[역할별 권한 매트릭스(O 허용 / - 불가)]")
print(mx.to_string())
print("승인(approve) 권한이 있는 역할:", [r for r in PERM if "approve" in PERM[r]], "-> 시스템·계약담당·계약관·법무에는 없음")

# ---- 이력 무결성과 이력 조회 ----
bad = wf.audit.verify()
print(f"\n감사 로그 {len(wf.audit.log)}건, 해시 체인 검증: {'정상' if bad is None else f'seq {bad}에서 불일치'} -> 이력 무결성 {'100%' if bad is None else '훼손'}(목표 100%: {'달성' if bad is None else '미달'})")
idx = 10
saved = json.dumps(wf.audit.log[idx], ensure_ascii=False)
wf.audit.log[idx]["reason"] = "사후 수정"
v = wf.audit.verify()
print(f"{idx}번째 기록 임의 수정 후 검증: {'위변조 탐지(seq ' + str(v) + ')' if v is not None else '탐지 실패'}")
wf.audit.log[idx] = json.loads(saved)
print("\n[이력 조회 화면] C02 건")
for e in wf.audit.log:
    if e.get("case") == "C02":
        print(f"  #{e['seq']:03d} {e['event']:15s} {e.get('role', e['who']):6s} v{e.get('version', 1)} | {e.get('reason', '')[:34]}")
print("※ 계약 조항의 최종 판단, 법무 검토, 계약 승인은 사람(계약관·법무·승인권자)이 하며, 시스템은 위험 표시와 수정 권고만 제공한다. 보안 규정과 결재선은 소속 기관 규정을 따른다.")

plt.figure(figsize=(6, 3.5)); EN = {"승인": "approved", "반려": "rejected", "종결(반려)": "closed-rejected", "보류(무응답)": "on hold"}
plt.bar([EN.get(k, k) for k in st], list(st.values()), color="tab:blue"); plt.title("HITL workflow outcomes (12 cases)"); plt.ylabel("cases"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[결재 결과] 12건 (모의 결재자 기준, 실제 결재는 사람이 한다)
  건     계약   최종 상태  버전                 비고
C01 C-0072 보류(무응답)   2 자동 확정 없음: 법무 응답 대기
C02 C-0099      승인   3            승인권자 결정
C03 C-0024      승인   3            승인권자 결정
C04 C-0067      승인   2            승인권자 결정
C05 C-0085      승인   2            승인권자 결정
C06 C-0022      승인   1            승인권자 결정
C07 C-0077      승인   2            승인권자 결정
C08 C-0040      승인   2            승인권자 결정
C09 C-0070      승인   2            승인권자 결정
C10 C-0065      승인   1            승인권자 결정
C11 C-0028      승인   2            승인권자 결정
C12 C-0007  종결(반려)   3      재기안 한도(2회) 초과
상태 집계: {'승인': 10, '보류(무응답)': 1, '종결(반려)': 1} / 반려 후 재기안 13회, 반려 이력 14건(모두 사유·근거 기록)

[권한 우회·절차 위반 시도]
  시스템 자동 승인               -> 차단: 권한 없음(시스템 역할에 approve 권한 없음)
  계약담당이 계약관 검토 대행         -> 차단: 권한 없음(계약담당 역할에 review_contract 권한 없음)
  검토 단계 건너뛰고 승인           -> 차단: 단계 위반(현재 상신, 필요 행위 review_contract)
  미등록 사용자 접근              -> 차단: 미등록 사용자
  사유 없는 검토                -> 차단: 사유·근거 입력 필수(반려 사유 포함)
  사유 없는 반려                -> 차단: 사유·근거 입력 필수(반려 사유 포함)
  감사 로그 삭제                -> 차단: 감사 로그 삭제·수정 권한은 누구에게도 없음
  감사관이 승인 시도              -> 차단: 권한 없음(감사관 역할에 approve 권한 없음)
  종결된 건 재승인               -> 차단: 이미 종결된 건(승인)
  존재하지 않는 건 승인            -> 차단: 존재하지 않는 건
  상신자가 본인 건 승인(분리 원칙)     -> 차단: 상신자와 승인자 분리 원칙 위반
  반려 건을 재기안 없이 검토 진행      -> 차단: 반려된 건은 재기안 후에만 진행 가능
  타인이 대신 재기안              -> 차단: 권한 없음(부서장 역할에 redraft 권한 없음)
우회·위반 시도 차단 13/13 (100%) (목표 100%: 달성)

[역할별 권한 매트릭스(O 허용 / - 불가)]
     제안 상신 계약관검토 법무검토 승인 반려 재기안 감사조회
시스템   O  -     -    -  -  -   -    -
계약담당  -  O     -    -  -  -   O    -
계약관   -  -     O    -  -  O   -    -
법무    -  -     -    O  -  O   -    -
승인권자  -  -     -    -  O  O   -    -
감사관   -  -     -    -  -  -   -    O
... (이하 생략)
```
</details>
